# Exercícios de Python e SQLite

**Aluna:** Giovanna Xavier

Neste notebook vamos praticar os principais comandos SQL usando o **SQLite** a partir do **Python**, com a biblioteca nativa `sqlite3`.

| Parte | Tema |
|---|---|
| 1 | Configuração e criação do banco (DDL) |
| 2 | Manipulação de dados (DML) |
| 3 | Agregação e ordenação |
| 4 | Transações, `COMMIT` e `ROLLBACK` |
| 5 | Joins e múltiplas tabelas |
| 6 | DCL controle de dados |

## **Parte 1:** Configuração e Criação do Banco de Dados

### 1.1 Importando bibliotecas

- `sqlite3` → biblioteca padrão do Python para trabalhar com SQLite.
- `pandas` → usada aqui para exibir os resultados das consultas em formato de tabela, mais fácil de ler.
- `os` → usada para apagar o arquivo do banco antes de começar, garantindo que o notebook sempre rode do zero.

In [1]:
import os
import sqlite3
import pandas as pd

### 1.2 Criação do banco de dados e conexão

`sqlite3.connect("empresa.db")` abre o arquivo do banco e **cria o arquivo se ele não existir**.

Antes de conectar, apagamos uma versão antiga do `empresa.db` (se houver). Assim, cada vez que o notebook é executado do início, os dados ficam sempre iguais e não acumulamos registros duplicados.

O `cursor` é o objeto que usamos para enviar comandos SQL ao banco.

In [36]:
# remove o banco antigo para que o notebook sempre comece do zero
if os.path.exists("empresa.db"):
    os.remove("empresa.db")

# conecta ao banco (cria o arquivo se não existir)
conn = sqlite3.connect("empresa.db")
cursor = conn.cursor()

# no SQLite, as chaves estrangeiras precisam ser ativadas em cada conexão
cursor.execute("PRAGMA foreign_keys = ON")
print("Banco empresa.db criado e conectado.")

Banco empresa.db criado e conectado.


#### Verificando a conexão

Para confirmar que a conexão funcionou, executamos uma consulta simples que pede ao banco a sua versão. Se o comando retornar um valor, a conexão está ativa

In [3]:
try:
    cursor.execute("SELECT sqlite_version()")
    versao = cursor.fetchone()[0]
    print(f"Conexão realizada com sucesso! Versão do SQLite: {versao}")
except sqlite3.Error as erro:
    print(f"Erro ao conectar: {erro}")

Conexão realizada com sucesso! Versão do SQLite: 3.50.4


#### Função auxiliar para exibir consultas

Para não repetir `cursor.execute(...)` + `print(cursor.fetchall())` em todas as consultas, criamos uma função que executa um `SELECT` e devolve o resultado como uma tabela do **pandas** (`DataFrame`), já com o nome das colunas.

In [ ]:
def consultar(sql):
    """execura um SELECT e retorna o resultado como df"""
    return pd.read_sql_query(sql, conn)

### 1.3 Criação de tabela (DDL)

**DDL (Data Definition Language)** são os comandos que definem a **estrutura** do banco. Aqui usamos `CREATE TABLE` para criar a tabela `funcionarios`:

| Coluna | Tipo | Regra |
|---|---|---|
| `id` | `INTEGER` | chave primária com incremento automático |
| `nome` | `TEXT` | não nulo |
| `cargo` | `TEXT` | não nulo |
| `salario` | `REAL` | — |
| `data_contratacao` | `TEXT` | não nulo |

> O SQLite não possui um tipo específico para datas. Por isso guardamos a data como `TEXT` no formato **`AAAA-MM-DD`**, que permite ordenar e comparar datas corretamente

In [5]:
cursor.execute('''
CREATE TABLE IF NOT EXISTS funcionarios (
    id               INTEGER PRIMARY KEY AUTOINCREMENT,
    nome             TEXT NOT NULL,
    cargo            TEXT NOT NULL,
    salario          REAL,
    data_contratacao TEXT NOT NULL
)
''')
conn.commit()
print("Tabela 'funcionarios' criada com sucesso!")

Tabela 'funcionarios' criada com sucesso!


### 1.4 Verificação da estrutura da tabela

O comando `PRAGMA table_info(nome_da_tabela)` mostra a estrutura da tabela: nome de cada coluna, tipo, se aceita nulo (`notnull`), valor padrão e se é chave primária (`pk`).

In [6]:
consultar("PRAGMA table_info(funcionarios)")

,cid,name,type,notnull,dflt_value,pk
0,0,id,INTEGER,0,None,1
1,1,nome,TEXT,1,None,0
2,2,cargo,TEXT,1,None,0
3,3,salario,REAL,0,None,0
4,4,data_contratacao,TEXT,1,None,0


## **Parte 2:** Manipulação de Dados (DML)

**DML (Data Manipulation Language)** são os comandos que **alteram os dados** dentro das tabelas: `INSERT`, `UPDATE` e `DELETE`

> Depois de cada alteração usamos `conn.commit()` para salvar as mudanças definitivamente no arquivo do banco

### 2.1 Inserção de dados `INSERT`

Inserimos 5 funcionários fictícios. Usamos `executemany` com `?` (*placeholders*): o Python preenche os valores de forma segura, evitando erros de aspas e ataques de **SQL Injection**.

In [7]:
funcionarios = [
    ('João Silva',     'Analista',      5000.00, '2026-01-15'),
    ('Maria Oliveira', 'Gerente',       8000.00, '2022-02-20'),
    ('Pedro Santos',   'Desenvolvedor', 6000.00, '2023-10-03'),
    ('Ana Costa',      'Designer',      5500.00, '2019-08-31'),
    ('Carlos Pereira', 'Analista',      5200.00, '2021-05-12'),
]

cursor.executemany('''
    INSERT INTO funcionarios (nome, cargo, salario, data_contratacao)
    VALUES (?, ?, ?, ?)
''', funcionarios)
conn.commit()

print(f"{cursor.rowcount} registros inseridos.")

5 registros inseridos.


### 2.2 Consulta de dados `SELECT`

`SELECT *` retorna **todas as colunas** de **todos os registros** da tabela

In [8]:
consultar("SELECT * FROM funcionarios")

,id,nome,cargo,salario,data_contratacao
0,1,João Silva,Analista,5000.0,2026-01-15
1,2,Maria Oliveira,Gerente,8000.0,2022-02-20
2,3,Pedro Santos,Desenvolvedor,6000.0,2023-10-03
3,4,Ana Costa,Designer,5500.0,2019-08-31
4,5,Carlos Pereira,Analista,5200.0,2021-05-12


### 2.3 Filtro com `WHERE`

A cláusula `WHERE` filtra os registros, retornando apenas os que atendem à condição

**Funcionários com salário superior a R$ 5.000:**

In [9]:
consultar("SELECT * FROM funcionarios WHERE salario > 5000")

,id,nome,cargo,salario,data_contratacao
0,2,Maria Oliveira,Gerente,8000.0,2022-02-20
1,3,Pedro Santos,Desenvolvedor,6000.0,2023-10-03
2,4,Ana Costa,Designer,5500.0,2019-08-31
3,5,Carlos Pereira,Analista,5200.0,2021-05-12


**Funcionários que trabalham como 'Gerente':**

In [10]:
consultar("SELECT * FROM funcionarios WHERE cargo = 'Gerente'")

,id,nome,cargo,salario,data_contratacao
0,2,Maria Oliveira,Gerente,8000.0,2022-02-20


### 2.4 Atualização de dados `UPDATE`

Aumentamos o salário do **João** em **10%**. Em vez de digitar o novo valor, fazemos o cálculo no próprio SQL: `salario * 1.10`

In [11]:
print("Antes:")
display(consultar("SELECT * FROM funcionarios WHERE nome = 'João Silva'"))

cursor.execute("UPDATE funcionarios SET salario = salario * 1.10 WHERE nome = 'João Silva'")
conn.commit()

print("Depois (+10%):")
consultar("SELECT * FROM funcionarios WHERE nome = 'João Silva'")

Antes:


,id,nome,cargo,salario,data_contratacao
0,1,João Silva,Analista,5000.0,2026-01-15


Depois (+10%):


,id,nome,cargo,salario,data_contratacao
0,1,João Silva,Analista,5500.0,2026-01-15


### 2.5 Exclusão de dados `DELETE`

Excluímos a funcionária **Ana Costa**. Assim como no `UPDATE`, o `WHERE` é essencial para não apagar a tabela inteira

In [12]:
cursor.execute("DELETE FROM funcionarios WHERE nome = 'Ana Costa'")
conn.commit()

# A linha excluída não aparece mais
consultar("SELECT * FROM funcionarios")

,id,nome,cargo,salario,data_contratacao
0,1,João Silva,Analista,5500.0,2026-01-15
1,2,Maria Oliveira,Gerente,8000.0,2022-02-20
2,3,Pedro Santos,Desenvolvedor,6000.0,2023-10-03
3,5,Carlos Pereira,Analista,5200.0,2021-05-12


### 2.6 Contagem de registros `COUNT`

In [13]:
consultar("SELECT COUNT(*) AS total_funcionarios FROM funcionarios")

,total_funcionarios
0,4


## **Parte 3:** Agregação e Ordenação

### 3.1 Funções de agregação (`SUM`, `AVG`, `MAX`, `MIN`)

**Soma total dos salários:**

In [14]:
consultar("SELECT SUM(salario) AS soma_salarios FROM funcionarios")

,soma_salarios
0,24700.0


**Salário médio da empresa** (`ROUND(..., 2)` arredonda para 2 casas decimais):

In [15]:
consultar("SELECT ROUND(AVG(salario), 2) AS salario_medio FROM funcionarios")

,salario_medio
0,6175.0


**Maior e menor salário:**

In [16]:
consultar("SELECT MAX(salario) AS maior_salario, MIN(salario) AS menor_salario FROM funcionarios")

,maior_salario,menor_salario
0,8000.0,5200.0


### 3.2 Agrupamento de dados `GROUP BY`

O `GROUP BY` junta os registros que têm o mesmo valor em uma coluna e aplica a função de agregação **para cada grupo**

**Salário médio por cargo:**

In [17]:
consultar('''
    SELECT cargo, ROUND(AVG(salario), 2) AS salario_medio
    FROM funcionarios
    GROUP BY cargo
''')

,cargo,salario_medio
0,Analista,5350.0
1,Desenvolvedor,6000.0
2,Gerente,8000.0


**Quantidade de funcionários por cargo:**

In [18]:
consultar('''
    SELECT cargo, COUNT(*) AS quantidade
    FROM funcionarios
    GROUP BY cargo
''')

,cargo,quantidade
0,Analista,2
1,Desenvolvedor,1
2,Gerente,1


### 3.3 Ordenação `ORDER BY`

**Funcionários do maior para o menor salário:**

In [19]:
consultar("SELECT * FROM funcionarios ORDER BY salario DESC")

,id,nome,cargo,salario,data_contratacao
0,2,Maria Oliveira,Gerente,8000.0,2022-02-20
1,3,Pedro Santos,Desenvolvedor,6000.0,2023-10-03
2,1,João Silva,Analista,5500.0,2026-01-15
3,5,Carlos Pereira,Analista,5200.0,2021-05-12


**Funcionários em ordem alfabética:**

In [20]:
consultar("SELECT * FROM funcionarios ORDER BY nome ASC")

,id,nome,cargo,salario,data_contratacao
0,5,Carlos Pereira,Analista,5200.0,2021-05-12
1,1,João Silva,Analista,5500.0,2026-01-15
2,2,Maria Oliveira,Gerente,8000.0,2022-02-20
3,3,Pedro Santos,Desenvolvedor,6000.0,2023-10-03


## **Parte 4:** Transações e ROLLBACK

Uma **transação** é um conjunto de operações que deve acontecer **por completo ou não acontecer de jeito nenhum** (princípio da *atomicidade*).

- `BEGIN` → inicia a transação
- `COMMIT` → confirma e salva as alterações permanentemente
- `ROLLBACK` → desfaz tudo que foi feito desde o `BEGIN`

> **Importante:** antes de um `BEGIN`, não pode haver outra transação aberta. Por isso fizemos `conn.commit()` após cada alteração nas partes anteriores

### 4.1 Simulação de transação com `ROLLBACK`

1. Iniciamos a transação;
2. Inserimos o funcionário **Lucas Almeida**;
3. Forçamos um erro com uma divisão por zero (`1 / 0`);
4. O erro cai no `except`, que chama `conn.rollback()` e desfaz a inserção.

In [21]:
try:
    cursor.execute("BEGIN")
    cursor.execute('''
        INSERT INTO funcionarios (nome, cargo, salario, data_contratacao)
        VALUES ('Lucas Almeida', 'Estagiário', 2000.00, '2024-03-01')
    ''')
    cursor.execute("SELECT COUNT(*) FROM funcionarios")
    print(f"Dentro da transação: {cursor.fetchone()[0]} funcionários")

    resultado = 1 / 0          # erro simulado -> ZeroDivisionError

    conn.commit()              # esta linha nunca é executada
except ZeroDivisionError as erro:
    conn.rollback()            # desfaz a inserção
    print(f"Erro: {erro} -> transação revertida com ROLLBACK")

Dentro da transação: 5 funcionários
Erro: division by zero -> transação revertida com ROLLBACK


**Verificação:** o Lucas **não** deve aparecer na tabela (resultado vazio)

In [22]:
consultar("SELECT * FROM funcionarios WHERE nome = 'Lucas Almeida'")

,id,nome,cargo,salario,data_contratacao


### 4.2 Transação com `COMMIT`

Agora repetimos a operação **sem** forçar o erro. Como tudo deu certo, o `conn.commit()` salva a inserção definitivamente

In [23]:
try:
    cursor.execute("BEGIN")
    cursor.execute('''
        INSERT INTO funcionarios (nome, cargo, salario, data_contratacao)
        VALUES ('Lucas Almeida', 'Estagiário', 2000.00, '2024-03-01')
    ''')
    conn.commit()              # salva permanentemente
    print("Transação confirmada com COMMIT!")
except sqlite3.Error as erro:
    conn.rollback()
    print(f"Erro: {erro} -> transação revertida")

Transação confirmada com COMMIT!


**Verificação:** agora o Lucas faz parte da tabela (apenas **uma** vez).

In [24]:
consultar("SELECT * FROM funcionarios")

,id,nome,cargo,salario,data_contratacao
0,1,João Silva,Analista,5500.0,2026-01-15
1,2,Maria Oliveira,Gerente,8000.0,2022-02-20
2,3,Pedro Santos,Desenvolvedor,6000.0,2023-10-03
3,5,Carlos Pereira,Analista,5200.0,2021-05-12
4,6,Lucas Almeida,Estagiário,2000.0,2024-03-01


## **Parte 5:** Joins e Múltiplas Tabelas

### 5.1 Criação da tabela `departamentos`

| Coluna | Tipo | Regra |
|---|---|---|
| `id` | `INTEGER` | chave primária com incremento automático |
| `nome_departamento` | `TEXT` | não nulo |
| `localizacao` | `TEXT` | — |

In [25]:
cursor.execute('''
CREATE TABLE IF NOT EXISTS departamentos (
    id                INTEGER PRIMARY KEY AUTOINCREMENT,
    nome_departamento TEXT NOT NULL,
    localizacao       TEXT
)
''')
conn.commit()

consultar("PRAGMA table_info(departamentos)")

,cid,name,type,notnull,dflt_value,pk
0,0,id,INTEGER,0,None,1
1,1,nome_departamento,TEXT,1,None,0
2,2,localizacao,TEXT,0,None,0


### 5.2 Inserindo os departamentos

In [26]:
departamentos = [
    ('Vendas',     'Bloco A'),
    ('RH',         'Bloco B'),
    ('TI',         'Bloco C'),
    ('Marketing',  'Bloco D'),
]
cursor.executemany(
    "INSERT INTO departamentos (nome_departamento, localizacao) VALUES (?, ?)",
    departamentos,
)
conn.commit()

consultar("SELECT * FROM departamentos")

,id,nome_departamento,localizacao
0,1,Vendas,Bloco A
1,2,RH,Bloco B
2,3,TI,Bloco C
3,4,Marketing,Bloco D


### 5.3 Adição de chave estrangeira (`ALTER TABLE`)

A coluna `departamento_id` é uma **chave estrangeira** (`REFERENCES departamentos(id)`): ela só pode receber valores que existam na coluna `id` da tabela `departamentos`. É ela que **liga** cada funcionário ao seu departamento.

In [27]:
cursor.execute('''
    ALTER TABLE funcionarios
    ADD COLUMN departamento_id INTEGER REFERENCES departamentos(id)
''')
conn.commit()

# A nova coluna aparece na estrutura da tabela
consultar("PRAGMA table_info(funcionarios)")

,cid,name,type,notnull,dflt_value,pk
0,0,id,INTEGER,0,None,1
1,1,nome,TEXT,1,None,0
2,2,cargo,TEXT,1,None,0
3,3,salario,REAL,0,None,0
4,4,data_contratacao,TEXT,1,None,0
5,5,departamento_id,INTEGER,0,None,0


A lista de chaves estrangeiras da tabela confirma a ligação com `departamentos`:

In [28]:
consultar("PRAGMA foreign_key_list(funcionarios)")

,id,seq,table,from,to,on_update,on_delete,match
0,0,0,departamentos,departamento_id,id,NO ACTION,NO ACTION,NONE


### 5.4 Atualização de dados

Associamos **cada funcionário** a um departamento usando `UPDATE` com `CASE` (funciona como um "se... então" dentro do SQL):

| Cargo | Departamento |
|---|---|
| Analista | 1 – Vendas |
| Gerente | 1 – Vendas |
| Desenvolvedor | 3 – TI |
| Estagiário | 3 – TI |

> O departamento **2 – RH** e o **4 – Marketing** ficaram sem funcionários, isso será útil para entender a diferença entre os tipos de JOIN

In [29]:
cursor.execute('''
    UPDATE funcionarios
    SET departamento_id = CASE
        WHEN cargo IN ('Analista', 'Gerente')        THEN 1
        WHEN cargo IN ('Desenvolvedor', 'Estagiário') THEN 3
    END
''')
conn.commit()

consultar("SELECT * FROM funcionarios")

,id,nome,cargo,salario,data_contratacao,departamento_id
0,1,João Silva,Analista,5500.0,2026-01-15,1
1,2,Maria Oliveira,Gerente,8000.0,2022-02-20,1
2,3,Pedro Santos,Desenvolvedor,6000.0,2023-10-03,3
3,5,Carlos Pereira,Analista,5200.0,2021-05-12,1
4,6,Lucas Almeida,Estagiário,2000.0,2024-03-01,3


### 5.5 `INNER JOIN`

In [30]:
consultar('''
    SELECT f.nome, f.cargo, d.nome_departamento
    FROM funcionarios f
    INNER JOIN departamentos d ON f.departamento_id = d.id
    ORDER BY f.nome
''')

,nome,cargo,nome_departamento
0,Carlos Pereira,Analista,Vendas
1,João Silva,Analista,Vendas
2,Lucas Almeida,Estagiário,TI
3,Maria Oliveira,Gerente,Vendas
4,Pedro Santos,Desenvolvedor,TI


### 5.6 `LEFT JOIN` com relação N para N

Um funcionário pode participar de **vários** projetos, e um projeto pode ter **vários** funcionários. Esse é um relacionamento **N para N (muitos para muitos)**.

Para representá-lo, criamos:

1. **`projetos`** → guarda os projetos (`id`, `nome_projeto`);
2. **`funcionarios_projetos`** → *tabela associativa* (ou intermediária) que só guarda os pares `funcionario_id` + `projeto_id`. A chave primária é a combinação das duas colunas, o que impede repetir o mesmo par

In [31]:
cursor.execute('''
CREATE TABLE IF NOT EXISTS projetos (
    id           INTEGER PRIMARY KEY AUTOINCREMENT,
    nome_projeto TEXT NOT NULL
)
''')

cursor.execute('''
CREATE TABLE IF NOT EXISTS funcionarios_projetos (
    funcionario_id INTEGER NOT NULL REFERENCES funcionarios(id),
    projeto_id     INTEGER NOT NULL REFERENCES projetos(id),
    PRIMARY KEY (funcionario_id, projeto_id)
)
''')
conn.commit()
print("Tabelas 'projetos' e 'funcionarios_projetos' criadas!")

Tabelas 'projetos' e 'funcionarios_projetos' criadas!


In [32]:
cursor.executemany(
    "INSERT INTO projetos (nome_projeto) VALUES (?)",
    [('Sistema de CRM',), ('Portal do Cliente',), ('App Mobile',)],
)

participacoes = [
    ('João Silva',     'Sistema de CRM'),
    ('Maria Oliveira', 'Sistema de CRM'),
    ('Pedro Santos',   'Portal do Cliente'),
    ('Pedro Santos',   'App Mobile'),
]
cursor.executemany('''
    INSERT INTO funcionarios_projetos (funcionario_id, projeto_id)
    VALUES (
        (SELECT id FROM funcionarios WHERE nome = ?),
        (SELECT id FROM projetos     WHERE nome_projeto = ?)
    )
''', participacoes)
conn.commit()

consultar("SELECT * FROM funcionarios_projetos")

,funcionario_id,projeto_id
0,1,1
1,2,1
2,3,2
3,3,3


Fazemos dois `LEFT JOIN` em sequência: `funcionarios` → `funcionarios_projetos` → `projetos`

In [33]:
consultar('''
    SELECT f.nome, p.nome_projeto
    FROM funcionarios f
    LEFT JOIN funcionarios_projetos fp ON f.id = fp.funcionario_id
    LEFT JOIN projetos p               ON fp.projeto_id = p.id
    ORDER BY f.nome
''')

,nome,nome_projeto
0,Carlos Pereira,NaN
1,João Silva,Sistema de CRM
2,Lucas Almeida,NaN
3,Maria Oliveira,Sistema de CRM
4,Pedro Santos,Portal do Cliente
5,Pedro Santos,App Mobile


**Comparação:** se usássemos `INNER JOIN` na mesma consulta, Carlos e Lucas **sumiriam** do resultado, porque não têm projeto. Essa é a principal diferença entre os dois tipos de JOIN

In [34]:
consultar('''
    SELECT f.nome, p.nome_projeto
    FROM funcionarios f
    INNER JOIN funcionarios_projetos fp ON f.id = fp.funcionario_id
    INNER JOIN projetos p               ON fp.projeto_id = p.id
    ORDER BY f.nome
''')

,nome,nome_projeto
0,João Silva,Sistema de CRM
1,Maria Oliveira,Sistema de CRM
2,Pedro Santos,Portal do Cliente
3,Pedro Santos,App Mobile


## **Parte 6:** DCL (Controle de Dados)

Os comandos SQL são divididos em grupos, de acordo com a sua finalidade:

| Sigla | Nome | Para que serve | Comandos |
|---|---|---|---|
| **DDL** | *Data Definition Language* | Define e altera a **estrutura** do banco (tabelas, colunas, índices) | `CREATE`, `ALTER`, `DROP`, `TRUNCATE` |
| **DML** | *Data Manipulation Language* | Insere, altera e apaga os **dados** dentro das tabelas | `INSERT`, `UPDATE`, `DELETE` |
| **DQL** | *Data Query Language* | **Consulta** os dados, sem modificá-los | `SELECT` |
| **DCL** | *Data Control Language* | Controla **quem pode acessar** e **o que cada usuário pode fazer** no banco | `GRANT`, `REVOKE` |

### Para que serve o DCL?

O **DCL** cuida da **segurança e das permissões** do banco de dados. Em sistemas com vários usuários (por exemplo, MySQL, PostgreSQL, SQL Server, Oracle), o administrador usa:

- **`GRANT`** → **concede** permissões a um usuário;
- **`REVOKE`** → **retira** permissões que foram concedidas.

Exemplos (sintaxe de PostgreSQL/MySQL):

```sql
-- O usuário 'analista_rh' pode apenas consultar a tabela funcionarios
GRANT SELECT ON funcionarios TO analista_rh;

-- O usuário 'gerente' pode consultar, inserir e atualizar
GRANT SELECT, INSERT, UPDATE ON funcionarios TO gerente;

-- Retira do 'analista_rh' a permissão de consulta
REVOKE SELECT ON funcionarios FROM analista_rh;
```

Com isso é possível aplicar o **princípio do menor privilégio**: cada pessoa recebe só o acesso de que precisa. Por exemplo, um estagiário pode ler dados, mas não pode ver salários nem apagar registros.

## Encerrando a conexão

Ao final, fechamos o cursor e a conexão com `close()`. Isso libera o arquivo do banco e os recursos usados pelo Python. Usamos `conn.commit()` antes, por segurança, para garantir que nada fique sem salvar.

In [35]:
conn.commit()
cursor.close()
conn.close()
print("Conexão com o banco de dados encerrada.")

Conexão com o banco de dados encerrada.
